# CTDC-Informed Exploitation-Type Prototype

This notebook mirrors the runnable script in `examples/run_ctdc_exploitation_type_prototype.py`: generate synthetic CTDC-style records, map them into standardized indicators, train an exploitation-type classifier, apply two triage scenarios, and build human-review queues.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.model_selection import train_test_split

from src.ai_core import ExploitationTypeClassifier, ExploitationTypeClassifierConfig
from src.data_mapping import CTDCMapper
from src.decision_layer import apply_triage_scenario, load_scenario
from src.human_review import build_review_queue_table
from src.utils import generate_ctdc_style_synthetic_records

In [ ]:
records = generate_ctdc_style_synthetic_records(n_records=400, random_state=42)
mapped = CTDCMapper().map_records(records)
train_df, local_df = train_test_split(
    mapped,
    test_size=200,
    random_state=42,
    stratify=mapped["exploitation_type"],
)
mapped.head()

In [ ]:
baseline = ExploitationTypeClassifier(
    ExploitationTypeClassifierConfig(model_type="logistic_regression")
).fit(train_df)

pretrained_module = ExploitationTypeClassifier(
    ExploitationTypeClassifierConfig(model_type="xgboost")
).fit(train_df)

scored = pretrained_module.score_records(local_df)
scored[["case_id", "P(Sex)", "P(Labor)", "P(Both)", "confidence", "predicted_exploitation_type"]].head()

In [ ]:
small_ngo = load_scenario(ROOT / "configs" / "scenario_small_ngo_multidisciplinary.yaml")
labor_task_force = load_scenario(ROOT / "configs" / "scenario_labor_task_force.yaml")

small_ngo_queue = build_review_queue_table(apply_triage_scenario(scored, small_ngo))
labor_queue = build_review_queue_table(apply_triage_scenario(scored, labor_task_force))

small_ngo_queue.head(10)

In [ ]:
labor_queue.head(10)